In [9]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

In [2]:
df = pd.read_csv("../data/moodcare_knowledge.csv")
df.head()

,id,category,user_state,advice,activity_type
0,1,loneliness,feeling alone even when surrounded by people,Feeling lonely can happen even when others are...,social_connection
1,2,loneliness,missing someone close,Missing someone is a natural feeling. You can ...,emotional_expression
2,3,loneliness,having no one to talk to,"When you feel like you have no one to talk to,...",social_support
3,4,loneliness,feeling disconnected from others,Feeling disconnected does not mean you are alo...,social_connection
4,5,loneliness,feeling lonely at night,Nighttime loneliness can feel stronger because...,self_care


In [3]:
print(df.shape)

(400, 5)


In [4]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
df['combined_text'] = ("Category: " + df['category'] + " | User State: " + df['user_state'] + " | Advice: " + df['advice'])

In [6]:
df[['combined_text']].head()

,combined_text
0,Category: loneliness | User State: feeling alo...
1,Category: loneliness | User State: missing som...
2,Category: loneliness | User State: having no o...
3,Category: loneliness | User State: feeling dis...
4,Category: loneliness | User State: feeling lon...


In [7]:
embedding = embedding_model.encode(df['combined_text'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [8]:
print(embedding.shape)

(400, 384)


In [10]:
print(chromadb.__version__)

1.5.9


In [11]:
client = chromadb.PersistentClient(path = "../vector_db")

In [12]:
collection = client.get_or_create_collection(name = "moodcare_knowledge")

In [13]:
print(collection.count())

0


In [14]:
if collection.count()==0:
    print("Collection is empty. Safe to insert data.")
else:
    print("Collection already contains data.")

Collection is empty. Safe to insert data.


In [16]:
if collection.count()==0:
    collection.add(
        ids = df['id'].astype(str).tolist(),
        documents = df['combined_text'].tolist(),
        embeddings = embedding.tolist(),
        metadatas = [
            {
                "category": row["category"],
                "activity_type" : row["activity_type"]
            }
            for _, row in df.iterrows()
        ]
    )
    print("Data inserted successfully!")
else:
    print("Collection already contains data. Skipping insertion.")

Data inserted successfully!


In [17]:
print(collection.count())

400


In [18]:
query = "I feel lonely and nobody understands me."

In [19]:
query_embedding = embedding_model.encode(query).tolist()

In [20]:
results = collection.query(
    query_embeddings = [query_embedding],
    n_results = 3
)

In [21]:
print(results)

{'ids': [['26', '8', '50']], 'embeddings': None, 'documents': [['Category: loneliness | User State: feeling lonely because of lack of attention | Advice: Feeling unnoticed can be painful. Try focusing on people who appreciate you and communicate your need for connection in a healthy way.', 'Category: loneliness | User State: feeling nobody understands me | Advice: Feeling misunderstood can be painful. Try expressing your thoughts clearly to someone you trust instead of keeping everything inside.', 'Category: loneliness | User State: feeling lonely and wanting someone to listen | Advice: Sometimes being heard is all you need. Try sharing your thoughts with someone supportive or write them down to understand your feelings better.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'loneliness', 'activity_type': 'communication'}, {'activity_type': 'communication', 'category': 'loneliness'}, {'activity_type': 'emotional_expressio

In [24]:
for i, doc in enumerate(results['documents'][0], start=1):
    print(f"Result {i}:")
    print(doc)
    print("-" * 80)

Result 1:
Category: loneliness | User State: feeling lonely because of lack of attention | Advice: Feeling unnoticed can be painful. Try focusing on people who appreciate you and communicate your need for connection in a healthy way.
--------------------------------------------------------------------------------
Result 2:
Category: loneliness | User State: feeling nobody understands me | Advice: Feeling misunderstood can be painful. Try expressing your thoughts clearly to someone you trust instead of keeping everything inside.
--------------------------------------------------------------------------------
Result 3:
Category: loneliness | User State: feeling lonely and wanting someone to listen | Advice: Sometimes being heard is all you need. Try sharing your thoughts with someone supportive or write them down to understand your feelings better.
--------------------------------------------------------------------------------


In [26]:
print(results.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [27]:
for i in range(len(results['documents'][0])):

    print("=" * 80)
    print(f"Result {i+1}:")

    print("\nID:")
    print(results["ids"][0][i])
    
    print("\nDistance:")
    print(results["distances"][0][i])
    
    print("\nMetdata:")
    print(results["metadatas"][0][i])
    
    print("\nDocument:")
    print(results["documents"][0][i])

Result 1:

ID:
26

Distance:
0.6782130599021912

Metdata:
{'category': 'loneliness', 'activity_type': 'communication'}

Document:
Category: loneliness | User State: feeling lonely because of lack of attention | Advice: Feeling unnoticed can be painful. Try focusing on people who appreciate you and communicate your need for connection in a healthy way.
Result 2:

ID:
8

Distance:
0.7247838973999023

Metdata:
{'activity_type': 'communication', 'category': 'loneliness'}

Document:
Category: loneliness | User State: feeling nobody understands me | Advice: Feeling misunderstood can be painful. Try expressing your thoughts clearly to someone you trust instead of keeping everything inside.
Result 3:

ID:
50

Distance:
0.7495507597923279

Metdata:
{'activity_type': 'emotional_expression', 'category': 'loneliness'}

Document:
Category: loneliness | User State: feeling lonely and wanting someone to listen | Advice: Sometimes being heard is all you need. Try sharing your thoughts with someone sup

In [28]:
def retrieve_context(query, top_k=3):
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings = [query_embedding],
        n_results = top_k)

    return results["documents"][0]

In [29]:
query = "I feel lonely and nobody understands me."

context = retrieve_context(query)

print(context)

['Category: loneliness | User State: feeling lonely because of lack of attention | Advice: Feeling unnoticed can be painful. Try focusing on people who appreciate you and communicate your need for connection in a healthy way.', 'Category: loneliness | User State: feeling nobody understands me | Advice: Feeling misunderstood can be painful. Try expressing your thoughts clearly to someone you trust instead of keeping everything inside.', 'Category: loneliness | User State: feeling lonely and wanting someone to listen | Advice: Sometimes being heard is all you need. Try sharing your thoughts with someone supportive or write them down to understand your feelings better.']
